# GEC Pipeline - Training with Star Labels

This notebook trains the GEC edit tagger using **star-compressed labels** (`labels_star`), which reduces the vocabulary from ~4500 to ~360 labels while preserving semantic meaning.

## Key Changes
- Uses `labels_star` for training (star-compressed: `K*`, `R_[abc*]`)
- More efficient learning with smaller label space
- Better generalization across variable-length edits

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

print("✓ Setup complete")

## 1. Load and Verify Data

In [ ]:
from src.services.gec.config import (
    CHECKPOINT_PATH,
    LABEL2ID_PATH,
    ID2LABEL_PATH,
)
import json

print("Checking data files...")
print(f"✓ Checkpoint exists: {CHECKPOINT_PATH.exists()}")
print(f"✓ Label mappings exist: {LABEL2ID_PATH.exists()}")

with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
    num_examples = sum(1 for _ in f)
print(f"✓ Training examples: {num_examples}")

with open(LABEL2ID_PATH, 'r', encoding='utf-8') as f:
    label2id = json.load(f)
print(f"✓ Label vocabulary: {len(label2id)} labels (star-compressed)")

In [ ]:
print("Sample training example:")
with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
    example = json.loads(f.readline().strip())
    
print(f"Subwords (first 10): {example['subwords'][:10]}")
print(f"Labels - count (first 10): {example['labels'][:10]}")
print(f"Labels - star (first 10): {example['labels_star'][:10]}")
print(f"\n✓ Using labels_star for training")

## 2. Setup Training

In [ ]:
from transformers import AutoTokenizer
from src.services.gec.training.datasets import GECTrainingDataset
from src.services.gec.training.model import GECTaggerModel
from src.services.gec.training.trainer import build_trainer

MODEL_CHECKPOINT = "aubmindlab/bert-base-arabertv02"
OUTPUT_DIR = Path("./gec_models/edit_tagger_v1")
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
MAX_LENGTH = 256

print(f"Model: {MODEL_CHECKPOINT}")
print(f"Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, Max len: {MAX_LENGTH}")

In [ ]:
with open(ID2LABEL_PATH, 'r', encoding='utf-8') as f:
    id2label = json.load(f)

print(f"Label vocabulary ({len(id2label)} labels):")
for i, (idx, label) in enumerate(list(id2label.items())[:30]):
    print(f"  {idx}: {label}")
if len(id2label) > 30:
    print(f"  ... and {len(id2label) - 30} more")

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✓ Vocab size: {tokenizer.vocab_size}")

In [ ]:
print("Loading dataset...")
train_dataset = GECTrainingDataset(
    jsonl_path=CHECKPOINT_PATH,
    tokenizer=tokenizer,
    label2id=label2id,
    max_length=MAX_LENGTH,
)
print(f"✓ Dataset size: {len(train_dataset)}")

sample = train_dataset[0]
print(f"\n✓ Sample verification:")
print(f"  input_ids: {len(sample['input_ids'])}")
print(f"  labels: {len(sample['labels'])}")
print(f"  Match: {len(sample['input_ids']) == len(sample['labels'])}")

In [ ]:
print("Initializing model...")
model = GECTaggerModel(
    checkpoint=MODEL_CHECKPOINT,
    label2id=label2id,
)
print(f"✓ Model ready ({len(label2id)} output labels)")

In [ ]:
print("Building trainer...")
trainer = build_trainer(
    model_wrapper=model,
    train_dataset=train_dataset,
    eval_dataset=None,
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    label2id_path=LABEL2ID_PATH,
    id2label_path=ID2LABEL_PATH,
)
print("✓ Trainer ready")

## 3. Train Model

In [ ]:
print("\n🚀 Starting training...")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("✓ Training complete")

In [ ]:
best_model_path = OUTPUT_DIR / "best"
print(f"Saving model to {best_model_path}...")

trainer.save_model(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

if best_model_path.exists():
    print(f"✓ Model saved")
    print(f"\nFiles:")
    for f in best_model_path.iterdir():
        print(f"  - {f.name}")
else:
    print("⚠️  Model save failed")

## Summary

✓ Trained with star-compressed labels (`labels_star`)  
✓ Label vocabulary: ~360 labels (vs ~4500 with count labels)  
✓ All dimension mismatches fixed  

### Next Steps
- Implement rewriter to apply predicted edits
- Add M² scorer evaluation
- Create proper dev/test splits